## Setup and Imports

In [ ]:
# Standard Library
import sys
import os 
import warnings
from pathlib import Path
warnings.filterwarnings("ignore", category=FutureWarning)

# Add src/ to path
project_root = Path().resolve().parents[1]
src_path = project_root / 'src'
if str(src_path) not in sys.path:
    sys.path.insert(0, str(src_path))
    
# Data manipulation
import pandas as pd
import numpy as np

# Machine Learning
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import silhouette_score

# Visualisation
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns

# Shared utilities
from utils.helpers import set_style, save_figure
set_style()

print("All imports successful!")
print(f"Project root: {project_root}")

## Load clean Master Dataset

In [ ]:
processed_path = project_root / 'data' / 'processed'

df_raw = pd.read_csv(
    processed_path / 'orders_clean_master.csv',
    parse_dates=['PURCHASE_TS', 'SHIP_TS']
)
df = df_raw.copy()
print("Data loaded successfully!")
print(f'Rows:    {len(df):,}')
print(f'Columns: {len(df.columns)}')
print(f'Date range: {df["PURCHASE_TS"].min()} → {df["PURCHASE_TS"].max()}')
print(f'Unique customers: {df["USER_ID"].nunique():,}')
print(f'Unique products:  {df["PRODUCT_NAME"].nunique()}')

In [ ]:
# Filter to analysis-ready orders with non-zero prices
if 'IS_ZERO_PRICE' in df.columns:
    df_analysis = df[df['IS_ZERO_PRICE'] == False].copy()
else:
    df_analysis = df[df['USD_PRICE'] > 0].copy()

print(f'Orders after removing $0 prices: {len(df_analysis):,}')
print(f'Customers in analysis set:        {df_analysis["USER_ID"].nunique():,}')

## Calculate RFM Scores

In [ ]:
# Reference date = one day after the last purchase in the dataset
# This means the most recent customer has recency = 1 day
# Using today's date would artificially inflate recency for all customers
reference_date = df_analysis['PURCHASE_TS'].max() + pd.Timedelta(days=1)
print(f'Reference date for recency calculation: {reference_date.date()}')
print(f'(One day after last order in dataset)')
print()

# Calculate RFM metrics per customer
rfm = df_analysis.groupby('USER_ID').agg(
    recency   = ('PURCHASE_TS',  lambda x: (reference_date - x.max()).days),
    frequency = ('ORDER_ID',     'count'),
    monetary  = ('USD_PRICE',    'sum')
).reset_index()

print(f'RFM table built for {len(rfm):,} customers')
print()
print('RFM Statistics')
print(rfm[['recency', 'frequency', 'monetary']].describe().round(2).to_string())

In [ ]:
# RFM score assignment
# We score each dimension 1-5 using quintiles (equal-sized groups)
# For Recency: LOWER days = HIGHER score (1 is bad, 5 is good)
# For Frequency and Monetary: HIGHER value = HIGHER score
#
# NOTE: Frequency is heavily skewed (91% of customers ordered once),
# so we use fixed thresholds instead of quintiles for F_score.

def qcut_score(series, q, ascending=True):
    """
    Score a series into q quantile buckets, handling duplicate edges.
    ascending=True  → higher value = higher score (Monetary)
    ascending=False → higher value = lower score  (Recency)
    """
    _, bin_edges = pd.qcut(series, q=q, retbins=True, duplicates='drop')
    n_bins = len(bin_edges) - 1

    if ascending:
        labels = list(range(1, n_bins + 1))
    else:
        labels = list(range(n_bins, 0, -1))

    return pd.cut(series, bins=bin_edges, labels=labels,
                  include_lowest=True).astype(int) # pyright: ignore[reportCallIssue]


def score_frequency(f):
    """
    Fixed threshold scoring for frequency.
    Quintiles won't work here — 91% of customers ordered exactly once,
    so equal-sized buckets collapse everything to score 1.
    Thresholds reflect the actual distribution: 1 / 2 / 3 / 4+ orders.
    """
    if f >= 4: return 5
    if f == 3: return 4
    if f == 2: return 3
    return 1  # f == 1 — majority of customers land here honestly


# Handle NaN in recency by assigning a high value (worst recency score)
max_recency = rfm['recency'].max()
rfm['recency'] = rfm['recency'].fillna(max_recency + 1)

rfm['R_score'] = qcut_score(rfm['recency'], q=5, ascending=False)
rfm['F_score'] = rfm['frequency'].apply(score_frequency)   # fixed thresholds, not qcut
rfm['M_score'] = qcut_score(rfm['monetary'], q=5, ascending=True)

# Normalise R and M to 1–5 in case fewer bins were produced (F is already 1-5 by design)
for col in ['R_score', 'M_score']:
    col_min, col_max = rfm[col].min(), rfm[col].max()
    if col_min != col_max:
        rfm[col] = ((rfm[col] - col_min) / (col_max - col_min) * 4 + 1).round().astype(int)

# Combined RFM score — average of all three dimensions (scale stays at 1-5)
rfm['RFM_score'] = (rfm['R_score'] + rfm['F_score'] + rfm['M_score']) / 3

print('RFM scores assigned ✓')
print()
print('Score distribution:')
print(f'  R_score range: {rfm["R_score"].min()} - {rfm["R_score"].max()}')
print(f'  F_score range: {rfm["F_score"].min()} - {rfm["F_score"].max()}')
print(f'  M_score range: {rfm["M_score"].min()} - {rfm["M_score"].max()}')
print(f'  RFM combined:  {rfm["RFM_score"].min():.2f} - {rfm["RFM_score"].max():.2f}')
print()
print('Sample of RFM table:')
print(rfm.head(10).to_string(index=False))

## K-Means Clustering

In [ ]:
# Normalise the three RFM dimensions
scaler = StandardScaler()
rfm_scaled = scaler.fit_transform(rfm[['recency', 'frequency', 'monetary']])

print('StandardScaler applied ✓')
print(f'Shape of scaled array: {rfm_scaled.shape}')
print(f'Mean after scaling (should be ~0): {rfm_scaled.mean(axis=0).round(4)}')
print(f'Std after scaling (should be ~1):  {rfm_scaled.std(axis=0).round(4)}')

In [ ]:
# Elbow method to find optimal k
# We run KMeans for each value of k and record:
#   inertia: sum of squared distances from each point to its cluster centre
#   silhouette: how well-separated clusters are (higher = better, max 1.0)

inertias    = []
silhouettes = []
k_range     = range(2, 11)

for k in k_range:
    km = KMeans(n_clusters=k, random_state=42, n_init=10)
    km.fit(rfm_scaled)
    inertias.append(km.inertia_)
    sil = silhouette_score(rfm_scaled, km.labels_)
    silhouettes.append(sil)
    print(f'  k={k}  inertia={km.inertia_:,.0f}  silhouette={sil:.3f}')

In [ ]:
# Plot elbow and silhouette charts
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Elbow chart
axes[0].plot(list(k_range), inertias, 'o-', color='#2E75B6', linewidth=2, markersize=6)
axes[0].set_title('Elbow method — choose k at the bend', fontsize=13, fontweight='medium')
axes[0].set_xlabel('Number of clusters (k)')
axes[0].set_ylabel('Inertia')
axes[0].yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{x:,.0f}'))
axes[0].grid(True, alpha=0.3)

# Silhouette chart
axes[1].plot(list(k_range), silhouettes, 'o-', color='#70AD47', linewidth=2, markersize=6)
axes[1].set_title('Silhouette score — higher is better', fontsize=13, fontweight='medium')
axes[1].set_xlabel('Number of clusters (k)')
axes[1].set_ylabel('Silhouette score')
axes[1].grid(True, alpha=0.3)

fig.suptitle('Choosing the optimal number of customer segments',
             fontsize=14, fontweight='medium', y=1.02)

save_figure(fig, '02_01_elbow_silhouette.png')
plt.show()

# Print recommendation
best_k = list(k_range)[silhouettes.index(max(silhouettes))]
print(f'Highest silhouette score: {max(silhouettes):.3f} at k={best_k}')
print(f'Recommendation: use k={best_k} clusters')

In [ ]:
# Apply final KMeans with chosen k
# We use k=4 as it typically produces the most interpretable business segments:
# Champions, Loyal, At Risk, and Lost/Inactive
# If your elbow/silhouette charts suggest a different k, change this value
CHOSEN_K = 4

km_final = KMeans(n_clusters=CHOSEN_K, random_state=42, n_init=10)
rfm['cluster'] = km_final.fit_predict(rfm_scaled)

print(f'K-Means applied with k={CHOSEN_K} ✓')
print()
print('Customers per cluster:')
print(rfm['cluster'].value_counts().sort_index().to_string())

## Segments Labelling

In [ ]:
# Profile each cluster by its average RFM values
# This tells us what kind of customers are in each group
cluster_profile = rfm.groupby('cluster').agg(
    customer_count = ('USER_ID',   'count'),
    avg_recency    = ('recency',   'mean'),
    avg_frequency  = ('frequency', 'mean'),
    avg_monetary   = ('monetary',  'mean'),
    avg_rfm_score  = ('RFM_score', 'mean')
).round(2)

print('Cluster profiles')
print('Use these to assign business labels:')
print('  Low recency + High frequency + High monetary = Champions')
print('  Low recency + Medium frequency + Medium monetary = Loyal')
print('  High recency + any = At Risk or Lost')
print()
print(cluster_profile.to_string())

In [ ]:
# Assign business labels based on cluster profiles 
# Read the cluster_profile output above and assign labels accordingly
# The cluster numbers (0,1,2,3) are arbitrary — KMeans assigns them randomly
# You need to look at avg_recency, avg_frequency, avg_monetary to decide
#
# CHAMPIONS:      low recency (bought recently), high frequency, high spend
# LOYAL:          moderate recency, above-average frequency and spend
# AT RISK:        used to buy well but recency is rising (they're drifting away)
# Lapsed:  high recency (haven't bought in a long time), low frequency

# Sort clusters by avg_rfm_score descending to identify them
ranked = cluster_profile['avg_rfm_score'].sort_values(ascending=False)
print('Clusters ranked by avg RFM score (highest = best customers):')
print(ranked)
print()

# Assign labels: rank 1 = Champions, rank 2 = Loyal, rank 3 = At Risk, rank 4 = Lapsed / Lost
label_map = {
    ranked.index[0]: 'Champions',
    ranked.index[1]: 'Loyal Customers',
    ranked.index[2]: 'At Risk',
    ranked.index[3]: 'Lapsed'
}

rfm['segment'] = rfm['cluster'].map(label_map)

print('Segment labels assigned:')
print(label_map)
print()
print('Customers per segment:')
print(rfm['segment'].value_counts())

In [ ]:
# Full segment profile with labels
segment_profile = rfm.groupby('segment').agg(
    customer_count = ('USER_ID',   'count'),
    pct_customers  = ('USER_ID',   lambda x: len(x) / len(rfm) * 100),
    avg_recency    = ('recency',   'mean'),
    avg_frequency  = ('frequency', 'mean'),
    avg_monetary   = ('monetary',  'mean'),
    total_revenue  = ('monetary',  'sum'),
    avg_rfm_score  = ('RFM_score', 'mean')
).round(2).sort_values('avg_rfm_score', ascending=False)

print('Final segment profiles')
print(segment_profile.to_string())

## Visualisations

In [ ]:
#Chart 1: Customer count and revenue share by segment
seg_colors = {
    'Champions':      '#1F4E79',
    'Loyal Customers':'#2E75B6',
    'At Risk':        '#ED7D31',
    'Lapsed':'#C00000'
}

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

seg_order  = ['Champions', 'Loyal Customers', 'At Risk', 'Lapsed']
seg_counts = [rfm[rfm['segment'] == s]['USER_ID'].count() for s in seg_order]
seg_rev    = [rfm[rfm['segment'] == s]['monetary'].sum() for s in seg_order]
colors_list = [seg_colors[s] for s in seg_order]

# Customer count
bars = axes[0].bar(seg_order, seg_counts, color=colors_list, alpha=0.85)
axes[0].set_title('Customer count by segment', fontsize=12, fontweight='medium')
axes[0].set_ylabel('Number of customers')
axes[0].yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{int(x):,}'))
for bar, count in zip(bars, seg_counts):
    axes[0].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 20,
                 f'{count:,}', ha='center', va='bottom', fontsize=10)
plt.setp(axes[0].xaxis.get_majorticklabels(), rotation=15, ha='right')

# Revenue share
total_rev = sum(seg_rev)
rev_pcts  = [r / total_rev * 100 if total_rev > 0 else 0 for r in seg_rev]

# Reverse order so highest bar appears at top
seg_order_r   = seg_order[::-1]
rev_pcts_r    = rev_pcts[::-1]
colors_list_r = colors_list[::-1]

hbars = axes[1].barh(seg_order_r, rev_pcts_r, color=colors_list_r, alpha=0.85, height=0.5)
axes[1].set_title('Revenue share by segment', fontsize=12, fontweight='medium')
axes[1].set_xlabel('Share of total revenue (%)')
axes[1].xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{x:.0f}%'))
axes[1].set_xlim(0, max(rev_pcts) * 1.15)  # headroom for labels

# Label each bar with % and absolute revenue
for bar, pct, rev in zip(hbars, rev_pcts_r, seg_rev[::-1]):
    label = f'{pct:.1f}%  (${rev:,.0f})' if pct > 0 else '0.0%  ($0)'
    axes[1].text(bar.get_width() + 0.5, bar.get_y() + bar.get_height()/2,
                 label, va='center', fontsize=9)

axes[1].spines[['top', 'right']].set_visible(False)

fig.suptitle('RFM customer segmentation overview',
             fontsize=14, fontweight='medium', y=1.02)
save_figure(fig, '02_01_segment_overview.png')
plt.show()

In [ ]:
# Chart 2: RFM dimension profiles per segment
# A grouped bar chart showing avg recency, frequency, and monetary per segment
# This makes it visually clear what makes each segment different

fig, axes = plt.subplots(1, 3, figsize=(16, 5))

metrics = [
    ('avg_recency',   'Average Recency (days)',  'Lower = more recent = better', '#C00000'),
    ('avg_frequency', 'Average Frequency (orders)', 'Higher = more loyal',       '#2E75B6'),
    ('avg_monetary',  'Average Spend ($)',       'Higher = more valuable',        '#70AD47'),
]

for ax, (metric, ylabel, note, color) in zip(axes, metrics):
    vals = [segment_profile.loc[s, metric] if s in segment_profile.index else 0
            for s in seg_order]
    bars = ax.bar(seg_order, vals, color=color, alpha=0.85)
    ax.set_title(ylabel, fontsize=11, fontweight='medium')
    ax.set_ylabel(ylabel)
    ax.text(0.5, -0.18, note, transform=ax.transAxes,
            ha='center', fontsize=9, color='gray', style='italic')
    for bar, val in zip(bars, vals):
        ax.text(bar.get_x() + bar.get_width()/2,
                bar.get_height() + max(vals) * 0.01,
                f'{val:.0f}', ha='center', va='bottom', fontsize=9)
    plt.setp(ax.xaxis.get_majorticklabels(), rotation=15, ha='right')

fig.suptitle('RFM dimension profiles by segment',
             fontsize=14, fontweight='medium', y=1.02)
save_figure(fig, '02_01_rfm_dimension_profiles.png')
plt.show()

In [ ]:
# Chart 3: RFM score distribution per segment — box plots
# Box plots show the spread within each segment, not just the average
# Wide boxes mean more variation within the segment

fig, ax = plt.subplots(figsize=(12, 5))

segment_order_plot = ['Champions', 'Loyal Customers', 'At Risk', 'Lapsed']
data_to_plot = [rfm[rfm['segment'] == s]['RFM_score'].values
                for s in segment_order_plot]

bp = ax.boxplot(data_to_plot, labels=segment_order_plot, patch_artist=True,
                medianprops={'color': 'white', 'linewidth': 2})

for patch, color in zip(bp['boxes'], [seg_colors[s] for s in segment_order_plot]):
    patch.set_facecolor(color)
    patch.set_alpha(0.75)

ax.set_title('RFM score distribution by segment',
             fontsize=13, fontweight='medium')
ax.set_ylabel('Combined RFM score (1-5)')
ax.set_xlabel('Segment')
ax.grid(axis='y', alpha=0.3)

save_figure(fig, '02_01_rfm_score_distribution.png')
plt.show()

In [ ]:
# Chart 4: Recency vs Monetary scatter — coloured by segment
# This is the most visually striking chart — it shows the full customer
# landscape in two dimensions at once

fig, ax = plt.subplots(figsize=(12, 6))

for segment in seg_order:
    subset = rfm[rfm['segment'] == segment]
    ax.scatter(
        subset['recency'],
        subset['monetary'],
        c=seg_colors[segment],
        label=f'{segment} (n={len(subset):,})',
        alpha=0.5,
        s=20,
        edgecolors='none'
    )

ax.set_title('Customer landscape: recency vs total spend\n(coloured by segment)',
             fontsize=13, fontweight='medium')
ax.set_xlabel('Recency (days since last purchase) — lower is more recent')
ax.set_ylabel('Total spend (USD)')
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'${x:,.0f}'))
ax.legend(loc='upper right', fontsize=9)
ax.grid(alpha=0.2)

save_figure(fig, '02_01_customer_landscape.png')
plt.show()

## Business Recommendations per Segment

In [ ]:
print('  SEGMENT BUSINESS RECOMMENDATIONS')
print()

recommendations = {
    'Champions': [
        '1,677 customers — 8.5% of base, 15.8% of revenue ($729,835)',
        'Higher order frequency (2.07 orders) but last seen ~444 days ago — they are slipping',
        'Re-engagement is worth pursuing — frequency history shows they can be repeat buyers',
        'Targeted incentive tied to their last product category to trigger a third purchase',
        'Do NOT discount broadly — reward repeat behaviour specifically (e.g. loyalty points)',
        'If reactivated, cross-sell higher ticket items — a Switch buyer is a natural PS5 prospect',
    ],
    'Loyal Customers': [
        '1,364 customers — 6.9% of base, 47.7% of revenue ($2,199,743)',
        'Highest monetary value by far (avg $1,613) but low frequency (1.13 orders) and last seen ~424 days ago',
        'These are big-ticket one-or-two-time buyers, not habitual loyals — treat them accordingly',
        'Personalised direct outreach justifiable given revenue concentration in this group',
        'Focus on a second purchase trigger — they spent big once, the relationship is not exhausted',
        'Do NOT assume retention — they have not returned in over a year and need a reason to come back',
    ],
    'At Risk': [
        '8,594 customers — 43.6% of base, 36.4% of revenue ($1,680,298)',
        'Paradoxically your most recently active group (243 days avg) despite the At Risk label',
        'Low avg spend ($196) and frequency (1.0) — volume is the value here, not depth',
        'Best candidate for a broad but low-cost re-engagement campaign — recency is on your side',
        'Single targeted incentive tied to last product category — do not over-personalise at this scale',
        'If no response within 30 days, move to Lapsed treatment',
    ],
    'Lapsed': [
        '8,088 customers — 41% of base, 24.5% of revenue ($1,493,608)',
        'Last purchased ~731 days ago — two years of silence confirms effective churn',
        'Low-cost single re-engagement email only — do not over-invest',
        'If no response after 2 attempts, suppress from all campaigns',
        'Analyse product category — many likely one-time console buyers with no natural repeat trigger',
        'Do NOT discount aggressively — reacquisition cost likely exceeds expected return',
    ]
}

for segment, actions in recommendations.items():
    seg_data = rfm[rfm['segment'] == segment]
    print(f'{segment.upper()}')
    print(f'  Customers: {len(seg_data):,}  |  '
          f'Avg recency: {seg_data["recency"].mean():.0f} days  |  '
          f'Avg spend: ${seg_data["monetary"].mean():.0f}  |  '
          f'Avg orders: {seg_data["frequency"].mean():.1f}')
    print()
    for action in actions:
        print(f'  → {action}')
    print()

## Export RFM Dataset

In [ ]:
os.makedirs(processed_path, exist_ok=True)

# Merge segment labels back onto the full orders data
# This lets downstream notebooks filter orders by customer segment
df_with_segments = df_analysis.merge(
    rfm[['USER_ID', 'recency', 'frequency', 'monetary',
         'R_score', 'F_score', 'M_score', 'RFM_score',
         'cluster', 'segment']],
    on='USER_ID', how='left'
)

# Export 1: Full RFM table (one row per customer)
rfm_output = processed_path / 'rfm_segments.csv'
rfm.to_csv(rfm_output, index=False)
print(f'Output 1: rfm_segments.csv')
print(f'  Rows: {len(rfm):,} (one per customer)')
print(f'  Columns: {list(rfm.columns)}')
print()

# Export 2: Orders with segment labels (one row per order)
orders_seg_output = processed_path / 'orders_with_segments.csv'
df_with_segments.to_csv(orders_seg_output, index=False)
print(f'Output 2: orders_with_segments.csv')
print(f'  Rows: {len(df_with_segments):,} (one per order)')
print()


print('Notebook 1 complete ✓')
print('Figures saved     → reports/figures/  (4 charts)')
print('Processed data    → data/processed/   (2 files)')
print('Ready for         → Notebook 2: BG/NBD CLV Modelling')
